# GeoClip Zero-Shot Baseline

Evaluate pretrained GeoClip on the MMlandmarks query set **without fine-tuning**.
The model embeds query ground images and gallery GPS coordinates into a shared
512-dim space, then retrieves the nearest GPS by cosine similarity.

**Gallery:** configurable via `gallery.source` in [configs/geoclip_baseline.yaml](../../configs/geoclip_baseline.yaml):
- `"paper"` (default, 100,539 coords = 99,539 index-satellite + 1,000 query-landmark GPS) — matches the camera-ready MML paper Sec 5.2 protocol. Reproduces the 21.37 % @1 km row of Table 3. Because every query's GT GPS is in the gallery, this is an **upper bound**.
- `"index"` (99,539 coords) — index-satellite only. Honest in-the-wild result (~6.67 % @1 km). Per the paper author: *"21 % is a geolocalization upper limit, 6.67 % is more realistic in the wild."*

**Queries:** 18,688 query ground images (multiple images per landmark, each scored against the landmark's ground-truth GPS).

**Metric:** Accuracy @ {1, 25, 200, 750, 2500} km (Haversine distance).

## 1. Setup

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml

plt.rcParams.update({"figure.dpi": 120})

# Load config
with open("../../configs/geoclip_baseline.yaml") as f:
    cfg = yaml.safe_load(f)

DATA_ROOT = Path("../../") / cfg["data"]["root"]
assert DATA_ROOT.exists(), f"DATA_ROOT not found: {DATA_ROOT}"

device = cfg["inference"]["device"] if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Data root: {DATA_ROOT.resolve()}")

Device: cuda
Data root: /dtu/blackhole/02/137570/MML


## 2. Load Model

In [2]:
from mmgeo.geolocalizations.geoclip.geoclip_baseline import (
    GeoClipBaseline,
    load_gallery_coords,
    load_query_data3,
    NewGeoClipBaseline
)
from mmgeo.geolocalizations.geoclip.evaluate import (
    accuracy_at_thresholds,
    median_error,
    haversine,
)

baseline = NewGeoClipBaseline(device=device,transformer=False)

total_params = sum(p.numel() for p in baseline.model.parameters())
trainable_params = sum(p.numel() for p in baseline.model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

/zhome/79/0/186934/Multimodal-Geo-Spatial-Learning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 10362.33it/s]

/zhome/79/0/186934/Multimodal-Geo-Spatial-Learning/.venv/lib/python3.11/site-packages/geoclip/model/location_encoder.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sel

Total parameters: 438,050,306
Trainable parameters: 10,432,257


## 3. Build GPS Gallery

In [3]:
# Load both galleries up-front; we rebuild per-source before each inference pass.
GALLERY_SOURCES = ["paper", "index"]
galleries = {}
for source in GALLERY_SOURCES:
    coords = load_gallery_coords(DATA_ROOT, source=source)
    galleries[source] = coords
    print(
        f"source={source!r:>8}: {len(coords):>6,} GPS points · "
        f"lat [{coords[:, 0].min():.2f}, {coords[:, 0].max():.2f}] · "
        f"lon [{coords[:, 1].min():.2f}, {coords[:, 1].max():.2f}]"
    )


source= 'paper': 100,539 GPS points · lat [20.00, 49.03] · lon [-155.89, -66.93]
source= 'index': 99,539 GPS points · lat [20.00, 49.03] · lon [-155.89, -66.93]


## 4. Load Query Data

In [4]:
thing = 1
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (1000, 1000, 1000)

[paper] build_gallery (100,539 GPS points)…


[paper] 1000 predictions in 29.3s

[index] build_gallery (99,539 GPS points)…


[index] 1000 predictions in 28.1s
 Threshold (km) paper (%) index (%)
              1     11.90      3.30
             25     24.80     21.20
            200     44.10     42.30
            750     71.80     71.60
           2500     93.60     93.80

 paper: median error 280.7 km · mean 639.0 km
 index: median error 306.5 km · mean 643.0 km


In [5]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (1902, 951, 1902)

[paper] build_gallery (100,539 GPS points)…


[paper] 951 predictions in 53.3s

[index] build_gallery (99,539 GPS points)…


[index] 951 predictions in 52.8s
 Threshold (km) paper (%) index (%)
              1     15.67      4.73
             25     29.34     24.71
            200     49.74     47.42
            750     77.71     76.97
           2500     95.27     95.27

 paper: median error 202.6 km · mean 507.1 km
 index: median error 233.6 km · mean 524.6 km


In [6]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (2583, 861, 2583)

[paper] build_gallery (100,539 GPS points)…


[paper] 861 predictions in 72.2s

[index] build_gallery (99,539 GPS points)…


[index] 861 predictions in 72.1s
 Threshold (km) paper (%) index (%)
              1     17.42      5.57
             25     31.36     25.90
            200     53.31     50.41
            750     81.53     80.84
           2500     96.75     96.40

 paper: median error 166.7 km · mean 433.9 km
 index: median error 197.9 km · mean 455.0 km


In [7]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (3104, 776, 3104)

[paper] build_gallery (100,539 GPS points)…


[paper] 776 predictions in 86.3s

[index] build_gallery (99,539 GPS points)…


[index] 776 predictions in 85.7s
 Threshold (km) paper (%) index (%)
              1     18.56      6.06
             25     32.73     27.84
            200     53.74     51.55
            750     81.96     81.19
           2500     96.52     95.62

 paper: median error 164.7 km · mean 428.3 km
 index: median error 190.3 km · mean 466.2 km


In [8]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (3490, 698, 3490)

[paper] build_gallery (100,539 GPS points)…


[paper] 698 predictions in 98.9s

[index] build_gallery (99,539 GPS points)…


[index] 698 predictions in 98.0s
 Threshold (km) paper (%) index (%)
              1     19.77      6.45
             25     35.96     30.37
            200     54.44     53.15
            750     82.81     82.23
           2500     96.13     96.13

 paper: median error 157.8 km · mean 434.9 km
 index: median error 173.7 km · mean 441.1 km


In [9]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (3738, 623, 3738)

[paper] build_gallery (100,539 GPS points)…


[paper] 623 predictions in 104.4s

[index] build_gallery (99,539 GPS points)…


[index] 623 predictions in 104.2s
 Threshold (km) paper (%) index (%)
              1     20.71      6.58
             25     38.04     30.98
            200     56.66     53.61
            750     82.83     80.58
           2500     95.99     95.02

 paper: median error 135.2 km · mean 419.5 km
 index: median error 169.1 km · mean 472.4 km


In [10]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4011, 573, 4011)

[paper] build_gallery (100,539 GPS points)…


[paper] 573 predictions in 111.9s

[index] build_gallery (99,539 GPS points)…


[index] 573 predictions in 112.1s
 Threshold (km) paper (%) index (%)
              1     22.34      6.81
             25     38.74     31.59
            200     58.46     55.32
            750     83.94     81.15
           2500     96.51     95.99

 paper: median error 119.3 km · mean 400.7 km
 index: median error 158.4 km · mean 451.7 km


In [11]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4184, 523, 4184)

[paper] build_gallery (100,539 GPS points)…


[paper] 523 predictions in 116.0s

[index] build_gallery (99,539 GPS points)…


[index] 523 predictions in 115.9s
 Threshold (km) paper (%) index (%)
              1     23.71      7.07
             25     41.11     32.70
            200     58.51     54.30
            750     83.56     81.26
           2500     95.60     95.22

 paper: median error 111.3 km · mean 419.7 km
 index: median error 158.0 km · mean 465.3 km


In [12]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4419, 491, 4419)

[paper] build_gallery (100,539 GPS points)…


[paper] 491 predictions in 122.5s

[index] build_gallery (99,539 GPS points)…


[index] 491 predictions in 122.2s
 Threshold (km) paper (%) index (%)
              1     24.85      6.92
             25     42.16     34.22
            200     58.04     53.97
            750     83.91     81.47
           2500     95.52     94.91

 paper: median error 107.1 km · mean 420.4 km
 index: median error 149.7 km · mean 464.4 km


In [13]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4550, 455, 4550)

[paper] build_gallery (100,539 GPS points)…


[paper] 455 predictions in 127.5s

[index] build_gallery (99,539 GPS points)…


[index] 455 predictions in 126.4s
 Threshold (km) paper (%) index (%)
              1     26.37      8.35
             25     43.30     35.16
            200     60.00     55.38
            750     83.30     80.22
           2500     95.16     94.29

 paper: median error 85.5 km · mean 416.9 km
 index: median error 143.4 km · mean 479.0 km


In [14]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4631, 421, 4631)

[paper] build_gallery (100,539 GPS points)…


[paper] 421 predictions in 129.1s

[index] build_gallery (99,539 GPS points)…


[index] 421 predictions in 128.6s
 Threshold (km) paper (%) index (%)
              1     27.55      8.79
             25     43.94     35.39
            200     59.86     55.11
            750     84.56     81.71
           2500     95.96     94.54

 paper: median error 80.7 km · mean 397.1 km
 index: median error 145.6 km · mean 463.3 km


In [15]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4752, 396, 4752)

[paper] build_gallery (100,539 GPS points)…


[paper] 396 predictions in 131.1s

[index] build_gallery (99,539 GPS points)…


[index] 396 predictions in 130.7s
 Threshold (km) paper (%) index (%)
              1     28.54      9.85
             25     44.95     37.88
            200     60.10     56.06
            750     84.09     81.06
           2500     95.71     93.94

 paper: median error 91.5 km · mean 412.0 km
 index: median error 142.1 km · mean 482.3 km


In [16]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4966, 382, 4966)

[paper] build_gallery (100,539 GPS points)…


[paper] 382 predictions in 139.3s

[index] build_gallery (99,539 GPS points)…


[index] 382 predictions in 139.8s
 Threshold (km) paper (%) index (%)
              1     29.06      9.69
             25     46.34     38.48
            200     60.47     55.50
            750     83.51     80.63
           2500     95.29     93.98

 paper: median error 67.6 km · mean 423.9 km
 index: median error 142.5 km · mean 486.8 km


In [17]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5096, 364, 5096)

[paper] build_gallery (100,539 GPS points)…


[paper] 364 predictions in 142.4s

[index] build_gallery (99,539 GPS points)…


[index] 364 predictions in 142.2s
 Threshold (km) paper (%) index (%)
              1     28.85      9.62
             25     45.33     37.64
            200     60.16     55.49
            750     84.34     82.14
           2500     95.05     94.78

 paper: median error 83.1 km · mean 422.7 km
 index: median error 142.0 km · mean 456.3 km


In [18]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5310, 354, 5310)

[paper] build_gallery (100,539 GPS points)…


[paper] 354 predictions in 145.7s

[index] build_gallery (99,539 GPS points)…


[index] 354 predictions in 144.8s
 Threshold (km) paper (%) index (%)
              1     28.25      9.89
             25     45.76     37.57
            200     59.04     53.95
            750     84.18     81.07
           2500     95.48     94.63

 paper: median error 83.1 km · mean 417.6 km
 index: median error 167.8 km · mean 475.1 km


In [19]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5328, 333, 5328)

[paper] build_gallery (100,539 GPS points)…


[paper] 333 predictions in 145.5s

[index] build_gallery (99,539 GPS points)…


[index] 333 predictions in 145.4s
 Threshold (km) paper (%) index (%)
              1     30.33      9.91
             25     46.55     37.24
            200     59.46     54.05
            750     84.08     79.58
           2500     95.20     93.39

 paper: median error 79.4 km · mean 401.1 km
 index: median error 161.0 km · mean 503.9 km


In [20]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5287, 311, 5287)

[paper] build_gallery (100,539 GPS points)…


[paper] 311 predictions in 145.7s

[index] build_gallery (99,539 GPS points)…


[index] 311 predictions in 145.4s
 Threshold (km) paper (%) index (%)
              1     30.87     10.29
             25     48.55     38.26
            200     61.09     56.27
            750     84.24     80.39
           2500     96.14     93.89

 paper: median error 55.9 km · mean 384.8 km
 index: median error 138.8 km · mean 485.8 km


In [21]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5328, 296, 5328)

[paper] build_gallery (100,539 GPS points)…


[paper] 296 predictions in 147.1s

[index] build_gallery (99,539 GPS points)…


[index] 296 predictions in 147.5s
 Threshold (km) paper (%) index (%)
              1     31.08     10.81
             25     49.32     39.19
            200     62.16     58.11
            750     83.45     80.41
           2500     95.61     94.26

 paper: median error 43.9 km · mean 394.7 km
 index: median error 134.4 km · mean 463.6 km


In [22]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5377, 283, 5377)

[paper] build_gallery (100,539 GPS points)…


[paper] 283 predictions in 147.2s

[index] build_gallery (99,539 GPS points)…


[index] 283 predictions in 147.1s
 Threshold (km) paper (%) index (%)
              1     31.80     11.31
             25     50.88     40.64
            200     62.54     57.60
            750     83.39     79.51
           2500     95.76     93.99

 paper: median error 19.6 km · mean 404.3 km
 index: median error 117.0 km · mean 490.9 km


In [23]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1
print(thing)

Query images: (5480, 274, 5480)

[paper] build_gallery (100,539 GPS points)…


[paper] 274 predictions in 151.2s

[index] build_gallery (99,539 GPS points)…


[index] 274 predictions in 150.5s
 Threshold (km) paper (%) index (%)
              1     32.48     11.68
             25     51.82     40.51
            200     63.50     56.57
            750     84.67     80.29
           2500     96.35     94.16

 paper: median error 17.5 km · mean 373.7 km
 index: median error 133.8 km · mean 482.9 km
21


## 5. Run Inference

## 6. Evaluate

## 7. Visualizations

## 8. Summary for Zero Shot

Zero-shot GeoClip on MMlandmarks, 18,688 query ground images, V100. Single HPC submit
runs both gallery sources back-to-back.

### `gallery.source: paper` — paper protocol (index + query = 100,539 GPS)

Reproduces the MML paper's Table 3 off-the-shelf GeoCLIP row within rounding. Because
every query's GT GPS sits in the gallery, this is an **upper bound** on achievable
performance, not an in-the-wild number.

| Threshold (km) | Accuracy (%) |
|---------------:|-------------:|
| 1              | 21.35        |
| 25             | 36.44        |
| 200            | 48.61        |
| 750            | 71.41        |
| 2500           | 91.52        |

- **Median error:** 225.2 km
- **Mean error:** 674.6 km

### `gallery.source: index` — honest in-the-wild (99,539 GPS, no query leakage)

Same model, query GT GPS removed from the gallery. Measures what off-the-shelf GeoCLIP
can actually do on US landmark localization without gallery leakage. Per Oskar
Kristoffersen (first author): *"21 % is a geolocalization upper limit, 6.67 % is more
realistic in the wild."*

| Threshold (km) | Accuracy (%) |
|---------------:|-------------:|
| 1              |  6.67        |
| 25             | 28.79        |
| 200            | 44.48        |
| 750            | 69.07        |
| 2500           | 91.07        |

- **Median error:** 294.3 km
- **Mean error:** 724.2 km

### Paper contrast

| Method | Dataset | Gallery | @1 km | @25 km | @200 km | @750 km | @2500 km |
|---|---|---:|---:|---:|---:|---:|---:|
| GeoClip (own paper) | Im2GPS3k (global) | 100k | 14.11 | 34.47 | 50.65 | 69.67 | 83.82 |
| Off-shelf GeoClip (MML paper) | MMlandmarks (US) | 101k (index+query) | **21.37** | **36.44** | 48.57 | 71.45 | 91.50 |
| **Ours (`paper`)** | MMlandmarks (US) | 101k (index+query) | **21.35** | **36.44** | 48.61 | 71.41 | 91.52 |
| **Ours (`index`)** | MMlandmarks (US) | 100k (index only) | **6.67** | **28.79** | 44.48 | 69.07 | 91.07 |

We reproduce the MML paper row to within 0.02 points at @1 km. The gap between the two
`Ours` rows is the query-leakage effect: including query GT coordinates in the gallery
gives the model a guaranteed-correct candidate to pick, inflating all thresholds.

> An earlier run on the 17,557 train-landmark gallery scored 19.22 % @1 km. That number
> is inflated by cluster-luck — train and query landmarks co-locate in the same tourist
> cities, so the nearest train-landmark GPS is often coincidentally <1 km from a query.
> Not a fair comparison to either of the paper galleries above.

**Next steps (Phase 2):** Fine-tune the Location Encoder and linear image head on the
MMlandmarks train split. Fair improvement lives on top of the `index` baseline
(28.79 % @25 km, not 36.44 %) — the paper gallery's leakage makes it hard to beat by
model changes alone.